# Notebook 6 — Hyperparameter Tuning

This notebook selects the top-performing baseline models from
Notebook 5 and tunes them using RandomizedSearchCV.

In [1]:
#Imports
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import time
import pandas as pd
import joblib

from src.preprocessing import create_preprocessor
from src.model_factory import build_models
from src.evaluation import evaluate_regression_model
from src.tuning import get_param_distributions

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [2]:
#Load Dataset & Recreate Split
DATA_DIR = Path("../data/processed")
df = pd.read_csv(DATA_DIR / "featured_train.csv")

X = df.drop(columns=["Price"])
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training Samples : {len(X_train)}")
print(f"Testing Samples  : {len(X_test)}")

Training Samples : 8369
Testing Samples  : 2093


In [3]:
#Load Baseline Results & Select Top Candidates
RESULTS_DIR = Path("../results")
baseline_results = pd.read_csv(RESULTS_DIR / "baseline_results.csv")

baseline_results = baseline_results.sort_values(
    by="R2 Score", ascending=False
).reset_index(drop=True)

TOP_N = 3
top_models = baseline_results.head(TOP_N)["Model"].tolist()

print(f"Tuning top {TOP_N} models: {top_models}")
baseline_results.head(TOP_N)

Tuning top 3 models: ['XGBoost', 'LightGBM', 'Random Forest']


,MAE,RMSE,R2 Score,Model
0,1200.56,1844.79,0.8368,XGBoost
1,1194.64,1885.63,0.8295,LightGBM
2,1178.04,1977.22,0.8125,Random Forest


In [4]:
#Build Preprocessor & Base Models
preprocessor = create_preprocessor(X_train)
all_models = build_models()

In [5]:
#Run RandomizedSearchCV for Each Top Model
start_time = time.time()

tuned_results = []
tuned_pipelines = {}

for model_name in top_models:

    print(f"\n{'=' * 60}")
    print(f"Tuning: {model_name}")
    print(f"{'=' * 60}")

    base_model = all_models[model_name]

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", base_model),
    ])

    param_distributions = get_param_distributions(model_name)

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=20,
        scoring="r2",
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )

    search.fit(X_train, y_train)

    best_pipeline = search.best_estimator_
    predictions = best_pipeline.predict(X_test)

    metrics = evaluate_regression_model(y_test, predictions)
    metrics["Model"] = model_name
    metrics["Best Params"] = search.best_params_

    tuned_results.append(metrics)
    tuned_pipelines[model_name] = best_pipeline

    print(
        f"\n{model_name:<20}"
        f"R²: {metrics['R2 Score']:.4f} | "
        f"RMSE: {metrics['RMSE']:.2f}"
    )

end_time = time.time()
print(f"\n\nTotal Tuning Time : {end_time - start_time:.2f} seconds")


Tuning: XGBoost
Fitting 5 folds for each of 20 candidates, totalling 100 fits

XGBoost             R²: 0.8412 | RMSE: 1819.60

Tuning: LightGBM
Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\arushi khare\OneDrive\Desktop\Flight-Price-Prediction\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



LightGBM            R²: 0.8220 | RMSE: 1926.26

Tuning: Random Forest
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Random Forest       R²: 0.8316 | RMSE: 1873.92


Total Tuning Time : 395.41 seconds


In [6]:
#Tuned Results Comparison
tuned_results_df = (
    pd.DataFrame(tuned_results)
    .sort_values(by="R2 Score", ascending=False)
    .reset_index(drop=True)
)

tuned_results_df[["Model", "MAE", "RMSE", "R2 Score"]].style.format({
    "MAE": "{:.2f}",
    "RMSE": "{:.2f}",
    "R2 Score": "{:.4f}"
})

,Model,MAE,RMSE,R2 Score
0,XGBoost,1171.42,1819.60,0.8412
1,Random Forest,1129.60,1873.92,0.8316
2,LightGBM,1162.70,1926.26,0.8220


In [7]:
#Best Overall Tuned Model
best_tuned_name = tuned_results_df.iloc[0]["Model"]
best_tuned_pipeline = tuned_pipelines[best_tuned_name]
best_tuned_params = tuned_results_df.iloc[0]["Best Params"]

print("=" * 60)
print(f"Best Tuned Model : {best_tuned_name}")
print(f"R² Score : {tuned_results_df.iloc[0]['R2 Score']:.4f}")
print(f"RMSE     : {tuned_results_df.iloc[0]['RMSE']:.2f}")
print("=" * 60)
print("\nBest Hyperparameters:")
for param, value in best_tuned_params.items():
    print(f"  {param}: {value}")

Best Tuned Model : XGBoost
R² Score : 0.8412
RMSE     : 1819.60

Best Hyperparameters:
  model__subsample: 1.0
  model__n_estimators: 200
  model__max_depth: 7
  model__learning_rate: 0.05
  model__colsample_bytree: 0.85


In [8]:
#Compare Against Baseline
baseline_r2 = baseline_results.loc[
    baseline_results["Model"] == best_tuned_name, "R2 Score"
].values[0]

tuned_r2 = tuned_results_df.iloc[0]["R2 Score"]
improvement = tuned_r2 - baseline_r2

print(f"Baseline R² ({best_tuned_name}) : {baseline_r2:.4f}")
print(f"Tuned R²    ({best_tuned_name}) : {tuned_r2:.4f}")
print(f"Improvement                    : {improvement:+.4f}")

if improvement > 0:
    print("\nTuning improved performance — saving tuned model as final.")
else:
    print("\nTuning did not improve performance — keeping baseline model.")

Baseline R² (XGBoost) : 0.8368
Tuned R²    (XGBoost) : 0.8412
Improvement                    : +0.0044

Tuning improved performance — saving tuned model as final.


In [9]:
#Save Final Tuned Model & Results
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    best_tuned_pipeline,
    MODEL_DIR / "final_tuned_model.pkl"
)

tuned_results_df.drop(columns=["Best Params"]).to_csv(
    RESULTS_DIR / "tuned_results.csv", index=False
)

print("Final tuned model saved successfully.")
print(f"Saved to: {MODEL_DIR / 'final_tuned_model.pkl'}")

Final tuned model saved successfully.
Saved to: ..\models\final_tuned_model.pkl
